In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xtrack as xt
import sys
helpers_path = f'../'
sys.path.insert(0, helpers_path)
from helpers_for_imperfections_model import find_elements, apply_errors, apply_girder_misalignments

In [2]:
# CHOOSE SEED HERE
seed = 4

### Load the reference line with correctors installed

In [3]:
print(f'Starting the process for seed {seed}!')

line_version = "LCC_106-2-3_z"

# Load reference line with correctors installed (note that we cycle it to the rf cavity)
line = xt.Line.from_json('lattices/reference_lattice_LCC_V106/line_fccee_p_ring_LCC_106-2-3_z_merged_dipoles_with_correctors.json')
line.cycle(name_first_element='rf400', inplace=True) # cycle to rf cavity
line.configure_radiation(model=None, model_beamstrahlung=None) # disable radiation 
line.twiss_default['method'] = '4d' # switch to 4d Twiss method
print(f'Loaded the {line_version} line, cycled it to the RF cavity, disabled radiation, and switched to 4D Twiss.')

tt = line.get_table()
tw_ref = line.twiss() #tw_ref has 4D Twiss

Starting the process for seed 4!


Loading line from dict: 100%|██████████| 15545/15545 [00:03<00:00, 4866.88it/s]


Done loading line from dict.           
Loaded the LCC_106-2-3_z line, cycled it to the RF cavity, disabled radiation, and switched to 4D Twiss.


### Grab all the markers separating the different lattice sections

In [4]:
# ARCS
arc_markers = [
    ('end_ds_start_arc_ipa', 'end_arc_start_ds_ipb'),
    ('end_ds_start_arc_ipb', 'end_arc_start_ds_ipd'),
    ('end_ds_start_arc_ipd', 'end_arc_start_ds_ipf'),
    ('end_ds_start_arc_ipf', 'end_arc_start_ds_ipg'),
    ('end_ds_start_arc_ipg', 'end_arc_start_ds_iph'),
    ('end_ds_start_arc_iph', 'end_arc_start_ds_ipj'),
    ('end_ds_start_arc_ipj', 'end_arc_start_ds_ipl'),
    ('end_ds_start_arc_ipl', 'end_arc_start_ds_ipa')]

# DISPERSION SUPPRESSION REGIONS (count as part of the arc)
DS_markers = [
    ('end_straight_start_ds_ipa', 'end_ds_start_arc_ipa'),
    ('end_arc_start_ds_ipb', 'end_ds_start_straight_ipb'),
    ('end_straight_start_ds_ipb', 'end_ds_start_arc_ipb'),
    ('end_arc_start_ds_ipd', 'end_ds_start_straight_ipd'),
    ('end_straight_start_ds_ipd', 'end_ds_start_arc_ipd'),
    ('end_arc_start_ds_ipf', 'end_ds_start_straight_ipf'),
    ('end_straight_start_ds_ipf', 'end_ds_start_arc_ipf'),
    ('end_arc_start_ds_ipg', 'end_ds_start_straight_ipg'),
    ('end_straight_start_ds_ipg', 'end_ds_start_arc_ipg'),
    ('end_arc_start_ds_iph', 'end_ds_start_straight_iph'),
    ('end_straight_start_ds_iph', 'end_ds_start_arc_iph'),
    ('end_arc_start_ds_ipj', 'end_ds_start_straight_ipj'),
    ('end_straight_start_ds_ipj', 'end_ds_start_arc_ipj'),
    ('end_arc_start_ds_ipl', 'end_ds_start_straight_ipl'),
    ('end_straight_start_ds_ipl', 'end_ds_start_arc_ipl'),
    ('end_arc_start_ds_ipa', 'end_ds_start_straight_ipa')]

# STRAIGHT SECTIONS
straight_markers = [
    ('end_ds_start_straight_ipb', 'end_straight_start_ds_ipb'),
    ('end_ds_start_straight_ipd', 'end_straight_start_ds_ipd'),
    ('end_ds_start_straight_ipf', 'end_straight_start_ds_ipf'),
    ('end_ds_start_straight_ipg', 'end_straight_start_ds_ipg'),
    ('end_ds_start_straight_iph', 'end_straight_start_ds_iph'),
    ('end_ds_start_straight_ipj', 'end_straight_start_ds_ipj'),
    ('end_ds_start_straight_ipl', 'end_straight_start_ds_ipl'),
    ('end_ds_start_straight_ipa', 'end_straight_start_ds_ipa')]

# Split the straight markers into INTERACTION REGION (IR) and TECHNICAL REGION (TR) markers
IR_markers = [
    ('end_ds_start_straight_ipa', 'end_straight_start_ds_ipa'),
    ('end_ds_start_straight_ipd', 'end_straight_start_ds_ipd'),
    ('end_ds_start_straight_ipg', 'end_straight_start_ds_ipg'),
    ('end_ds_start_straight_ipj', 'end_straight_start_ds_ipj')]

TR_markers = [
    ('end_ds_start_straight_ipb', 'end_straight_start_ds_ipb'),
    ('end_ds_start_straight_ipf', 'end_straight_start_ds_ipf'),
    ('end_ds_start_straight_iph', 'end_straight_start_ds_iph'),
    ('end_ds_start_straight_ipl', 'end_straight_start_ds_ipl')]

In [ ]:
# # Save the markers to a txt file
# with open(f"lattices/reference_lattice_LCC_V106/marker_pairs_{line_version}.txt", "w") as f:
#     f.write("ARC_MARKERS:\n")
#     for marker in arc_markers:
#         f.write(f"{marker}\n")
#     f.write("\nDS_MARKERS:\n")
#     for marker in DS_markers:
#         f.write(f"{marker}\n")
#     f.write("\nSTRAIGHT_MARKERS:\n")
#     for marker in straight_markers:
#         f.write(f"{marker}\n")
#     f.write("\nIR_MARKERS:\n")
#     for marker in IR_markers:
#         f.write(f"{marker}\n")
#     f.write("\nTR_MARKERS:\n")
#     for marker in TR_markers:
#         f.write(f"{marker}\n")

### Filter out arc elements (marker-based filtering)

In [7]:
arc_quad_names = find_elements(tt, marker_pairs=[arc_markers, DS_markers], element_type='Quadrupole').name # note: include the ds elements with the arc elements
arc_dipole_names = find_elements(tt, marker_pairs=[arc_markers, DS_markers], element_type='RBend').name
arc_sext_names = find_elements(tt, marker_pairs=[arc_markers, DS_markers], element_type='Sextupole').name

### Apply imperfections to arc elements

In [8]:
# --------------------------- Misaligments in arcs --------------------------- #

# ARC DIPOLES
apply_errors(line=line, pattern=None, seed=seed, sigmas=[1e-3, 1e-3, 0.5e-3, 1e-3], # <-- chosen error tolerances
             attrs=['shift_x', 'shift_y', 'shift_s', 'rot_s_rad_no_frame'], # <-- attributes to apply the errors to
             switch_name='on_misalignment_dip_arc', # <-- switch to turn on/off the errors
             element_names=arc_dipole_names) # <-- lattice elements to apply the errors to

line.vars['on_misalignment_dip_arc'] = 0

# ARC QUADRUPOLES
apply_errors(line=line, pattern=None, seed=seed, sigmas=[50e-6, 50e-6, 100e-6, 50e-6], # <-- chosen error tolerances
             attrs=['shift_x', 'shift_y', 'shift_s', 'rot_s_rad_no_frame'], # <-- attributes to apply the errors to
             switch_name='on_misalignment_quad_arc', # <-- switch to turn on/off the errors
             element_names=arc_quad_names) # <-- lattice elements to apply the errors to

line.vars['on_misalignment_quad_arc'] = 0

# ARC SEXTUPOLES
apply_errors(line=line, pattern=None, seed=seed, sigmas=[50e-6, 50e-6, 100e-6, 50e-6], # <-- chosen error tolerances
             attrs=['shift_x', 'shift_y', 'shift_s', 'rot_s_rad_no_frame'], # <-- attributes to apply the errors to
             switch_name='on_misalignment_sext_arc', # <-- switch to turn on/off the errors
             element_names=arc_sext_names) # <-- lattice elements to apply the errors to

line.vars['on_misalignment_sext_arc'] = 0

print(f'Finished installing all arc misalignments!')

# --------------------------- Field errors in arcs --------------------------- #

# ARC DIPOLES
_ = apply_errors(line=line, pattern=None, seed=seed, sigmas=[1e-3], attrs=['k0'], # <-- chosen error tolerance and attribute to apply the errors to
                    wrt_current_expr=True, # <-- careful, do not apply it twice!
                    apply_relative=True, # <-- to apply the value as a relative change (i.e., multiplied by the existing value)
                    switch_name='on_field_error_dip_arc', # <-- switch to turn on/off the errors
                    element_names=arc_dipole_names) # <-- lattice elements to apply the errors to

line.vars['on_field_error_dip_arc'] = 0

# ARC QUADRUPOLES
_ = apply_errors(line=line, pattern=None, seed=seed, sigmas=[2e-4], attrs=['k1'], # <-- chosen error tolerance and attribute to apply the errors to
                    wrt_current_expr=True, # <-- careful, do not apply it twice!
                    apply_relative=True, # <-- to apply the value as a relative change (i.e., multiplied by the existing value)
                    switch_name='on_field_error_quad_arc', # <-- switch to turn on/off the errors
                    element_names=arc_quad_names) # <-- lattice elements to apply the errors to

line.vars['on_field_error_quad_arc'] = 0

# ARC SEXTUPOLES
_ = apply_errors(line=line, pattern=None, seed=seed, sigmas=[2e-4], attrs=['k2'], # <-- chosen error tolerance and attribute to apply the errors to
                    wrt_current_expr=True, # <-- careful, do not apply it twice!
                    apply_relative=True, # <-- to apply the value as a relative change (i.e., multiplied by the existing value)
                    switch_name='on_field_error_sext_arc', # <-- switch to turn on/off the errors
                    element_names=arc_sext_names) # <-- lattice elements to apply the errors to

line.vars['on_field_error_sext_arc'] = 0

print(f'Finished installing all arc field errors!')

Finished installing all arc misalignments!
Finished installing all arc field errors!


### Apply imperfections to girders in arcs

In [9]:
# Girder misalignments
groups = apply_girder_misalignments(line=line, seed=seed, sigmas=[0.15e-3, 0.15e-3, 0.5e-3, 0.15e-3], # <-- chosen error tolerances
                                        attrs=['shift_x', 'shift_y', 'shift_s', 'rot_s_rad_no_frame'], # <-- attributes to apply the errors to
                                        switch_name='on_misalignment_girder', # <-- switch to turn on/off the errors
                                        line_table=tt, marker_pairs_arcs=[arc_markers]) # <-- filtering via marker pairs to identify the arcs

line.vars['on_misalignment_girder'] = 0

print(f'Finished installing all girder misalignments!')

### Filter our straight section elements (using marker pairs and regex patterns)

In [10]:
IR_dipole_names = find_elements(tt, marker_pairs=[IR_markers], element_type='RBend').name # interaction region dipoles
TR_dipole_names = find_elements(tt, marker_pairs=[TR_markers], element_type='RBend').name # technical region dipoles
IR_quad_names = find_elements(tt, marker_pairs=[IR_markers], element_type='Quadrupole').name # interaction region quadrupoles
FD_quad_names = find_elements(tt, pattern=['qd0a', 'qd0b', 'qd0cr', 'qd0cl', 'qf1a', 'qf1b', 'qf1cr', 'qf1cl', 'qf1dr', 'qf1dl'], element_type='Quadrupole').name # final doublet quadrupoles
FF_quad_names = find_elements(tt, marker_pairs=[IR_markers], element_type='Quadrupole', except_names=FD_quad_names).name # final focusing quadrupoles
TR_quad_names = find_elements(tt, marker_pairs=[TR_markers], element_type='Quadrupole').name # technical region quadrupoles
IR_sext_names = find_elements(tt, marker_pairs=[IR_markers], element_type='Sextupole').name # interaction region sextupoles

### Apply imperfections to straight section elements

In [11]:
# ------------------------- Misaligments in straights ------------------------ #

# INTERACTION REGION (IR) DIPOLES
apply_errors(line=line, pattern=None, seed=seed, sigmas=[1e-3, 1e-3, 0.1e-3, 1e-3], 
             attrs=['shift_x', 'shift_y', 'shift_s', 'rot_s_rad_no_frame'], 
             switch_name='on_misalignment_dip_ir',
             element_names=IR_dipole_names)

line.vars['on_misalignment_dip_ir'] = 0

# TECHNICAL REGION (TR) DIPOLES
apply_errors(line=line, pattern=None, seed=seed, sigmas=[1e-3, 1e-3, 0.5e-3, 1e-3], 
             attrs=['shift_x', 'shift_y', 'shift_s', 'rot_s_rad_no_frame'], 
             switch_name='on_misalignment_dip_tr',
             element_names=TR_dipole_names)

line.vars['on_misalignment_dip_tr'] = 0

# FINAL DOUBLET (FD) QUADRUPOLES
apply_errors(line=line, pattern=None, seed=seed, sigmas=[30e-6, 30e-6, 100e-6, 30e-6], 
             attrs=['shift_x', 'shift_y', 'shift_s', 'rot_s_rad_no_frame'], 
             switch_name='on_misalignment_quad_fd',
             element_names=FD_quad_names)

line.vars['on_misalignment_quad_fd'] = 0

# FINAL FOCUSING (FF) QUADRUPOLES
apply_errors(line=line, pattern=None, seed=seed, sigmas=[30e-6, 30e-6, 100e-6, 30e-6], 
             attrs=['shift_x', 'shift_y', 'shift_s', 'rot_s_rad_no_frame'], 
             switch_name='on_misalignment_quad_ff',
             element_names=FF_quad_names)

line.vars['on_misalignment_quad_ff'] = 0

# TECHNICAL REGION (TR) QUADRUPOLES
apply_errors(line=line, pattern=None, seed=seed, sigmas=[100e-6, 100e-6, 100e-6, 100e-6], 
             attrs=['shift_x', 'shift_y', 'shift_s', 'rot_s_rad_no_frame'], 
             switch_name='on_misalignment_quad_tr',
             element_names=TR_quad_names)

line.vars['on_misalignment_quad_tr'] = 0

# INTERACTION REGION (IR) SEXTUPOLES
apply_errors(line=line, pattern=None, seed=seed, sigmas=[30e-6, 30e-6, 100e-6, 30e-6], 
             attrs=['shift_x', 'shift_y', 'shift_s', 'rot_s_rad_no_frame'], 
             switch_name='on_misalignment_sext_ir',
             element_names=IR_sext_names)

line.vars['on_misalignment_sext_ir'] = 0

print (f'Finished installing all straight section misalignments!')

# ------------------------- Field errors in straights ------------------------ #

# INTERACTION REGION (IR) DIPOLES
_ = apply_errors(line=line, pattern=None, seed=seed, 
                    sigmas=[1e-3], attrs=['k0'], wrt_current_expr=True, # <-- careful, do not apply it twice!
                    apply_relative=True, # <-- to apply the value as a relative change (i.e., multiplied by the existing value)
                    switch_name='on_field_error_dip_ir',
                    element_names=IR_dipole_names)

line.vars['on_field_error_dip_ir'] = 0

# TECHNICAL REGION (TR) DIPOLES
_ = apply_errors(line=line, pattern=None, seed=seed, 
                    sigmas=[1e-3], attrs=['k0'], wrt_current_expr=True, # <-- careful, do not apply it twice!
                    apply_relative=True, # <-- to apply the value as a relative change (i.e., multiplied by the existing value)
                    switch_name='on_field_error_dip_tr',
                    element_names=TR_dipole_names)

line.vars['on_field_error_dip_tr'] = 0

# FINAL DOUBLET (FD) QUADRUPOLES
_ = apply_errors(line=line, pattern=None, seed=seed, 
                    sigmas=[0.5e-4], attrs=['k1'], wrt_current_expr=True, # <-- careful, do not apply it twice!
                    apply_relative=True, # <-- to apply the value as a relative change (i.e., multiplied by the existing value)
                    switch_name='on_field_error_quad_fd',
                    element_names=FD_quad_names)

line.vars['on_field_error_quad_fd'] = 0

# FINAL FOCUSING (FF) QUADRUPOLES
_ = apply_errors(line=line, pattern=None, seed=seed, 
                    sigmas=[1e-4], attrs=['k1'], wrt_current_expr=True, # <-- careful, do not apply it twice!
                    apply_relative=True, # <-- to apply the value as a relative change (i.e., multiplied by the existing value)
                    switch_name='on_field_error_quad_ff',
                    element_names=FF_quad_names)

line.vars['on_field_error_quad_ff'] = 0

# TECHNICAL REGION (TR) QUADRUPOLES
_ = apply_errors(line=line, pattern=None, seed=seed, 
                    sigmas=[2e-4], attrs=['k1'], wrt_current_expr=True, # <-- careful, do not apply it twice!
                    apply_relative=True, # <-- to apply the value as a relative change (i.e., multiplied by the existing value)
                    switch_name='on_field_error_quad_tr',
                    element_names=TR_quad_names)

line.vars['on_field_error_quad_tr'] = 0

# INTERACTION REGION (IR) SEXTUPOLES
_ = apply_errors(line=line, pattern=None, seed=seed, 
                    sigmas=[1e-4], attrs=['k2'], wrt_current_expr=True, # <-- careful, do not apply it twice!
                    apply_relative=True, # <-- to apply the value as a relative change (i.e., multiplied by the existing value)
                    switch_name='on_field_error_sext_ir',
                    element_names=IR_sext_names)

line.vars['on_field_error_sext_ir'] = 0

print(f'Finished installing all straight section field errors!')

Finished installing all straight section misalignments!
Finished installing all straight section field errors!


In [12]:
# Save line with imperfections switches installed
line.to_json(f'lattices/lattices_with_imperfections/{line_version}_line_with_imperfections_switches_seed{seed}.json')
print(f'Line with imperfections switches saved to {line_version}_line_with_imperfections_switches_seed{seed}.json')

Line with imperfections switches saved to LCC_106-2-3_z_line_with_imperfections_switches_seed4.json


In [13]:
# Check the switch names
for switch_name in line.vars.keys():
    if 'on_' in switch_name:
        print(line.vars[switch_name])

vars['rf_harmon_400']
vars['on_qno_corrector']
vars['on_qsk_corrector']
vars['on_misalignment_dip_arc']
vars['on_misalignment_quad_arc']
vars['on_misalignment_sext_arc']
vars['on_field_error_dip_arc']
vars['on_field_error_quad_arc']
vars['on_field_error_sext_arc']
vars['on_misalignment_girder']
vars['on_misalignment_dip_ir']
vars['on_misalignment_dip_tr']
vars['on_misalignment_quad_fd']
vars['on_misalignment_quad_ff']
vars['on_misalignment_quad_tr']
vars['on_misalignment_sext_ir']
vars['on_field_error_dip_ir']
vars['on_field_error_dip_tr']
vars['on_field_error_quad_fd']
vars['on_field_error_quad_ff']
vars['on_field_error_quad_tr']
vars['on_field_error_sext_ir']
